### 前复权数据

In [1]:
import akshare as ak
import pandas as pd
import talib
import numpy as np
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

/home/ts/.local/share/virtualenvs/jupyter.13-jNpHegMS/lib/python3.13/site-packages/py_mini_racer/py_mini_racer.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


#### 除权日期

In [ ]:
ak.stock_fhps_detail_em(symbol="300303")['除权除息日'].astype(str)

In [ ]:
dfcq = ak.stock_fhps_detail_em(symbol="000001")['除权除息日'].astype(str)

In [ ]:
dat = ak.stock_fhps_detail_ths(symbol="600256")['A股除权除息日'].dropna().reset_index(drop=True)

In [ ]:
# 个股
ak.stock_zh_a_hist(symbol="000001", period="daily", start_date="20000101", end_date='20251124', adjust="qfq") # 东财 (该接口数据质量高, 访问无限制)
ak.stock_zh_a_daily(symbol="sz000001", start_date="20000101", end_date="20251124", adjust="qfq")
# 指数
ak.index_zh_a_hist(symbol="000001", period="daily", start_date="20000101", end_date="20251124")
# 复权因子
ak.stock_zh_a_daily(symbol="sz000002", adjust="hfq-factor")
ak.stock_zh_a_daily(symbol="sh600006", adjust="qfq-factor")

In [ ]:
dd = ak.stock_zh_a_hist(symbol="000001", period="daily", start_date="20000101", end_date='20300101', adjust="qfq")

In [ ]:
df = ak.stock_zh_a_daily(symbol="sz000001", start_date="20000101", end_date="20300101", adjust="qfq")

In [ ]:
df.info()

In [ ]:
def create_features(df: pd.DataFrame, 
                                 price_cols: Dict[str, str] = None,
                                 volume_col: str = 'vol',
                                 date_col: str = 'datetime',
                                 feature_groups: List[str] = None) -> pd.DataFrame:
    """
    构建A股技术指标特征（优化版本）
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含股票数据的DataFrame，至少包含价格和成交量数据
    price_cols : dict, optional
        价格列的映射
    volume_col : str, default 'volume'
        成交量列名
    date_col : str, default 'date'
        日期列名
    feature_groups : list, optional
        需要构建的特征组
    
    Returns:
    --------
    pd.DataFrame
        包含技术指标特征的DataFrame
    """
    
    # 复制数据避免修改原始数据
    # data = df.copy()
    data = df[['date', 'open', 'high', 'low', 'close']].copy()
    
    # 自动识别价格列
    if price_cols is None:
        possible_names = {
            'open': ['open', 'Open', 'OPEN', 'o', 'O'],
            'high': ['high', 'High', 'HIGH', 'h', 'H'],
            'low': ['low', 'Low', 'LOW', 'l', 'L'],
            'close': ['close', 'Close', 'CLOSE', 'c', 'C']
        }
        
        price_cols = {}
        for key, possible_values in possible_names.items():
            for col in possible_values:
                if col in data.columns:
                    price_cols[key] = col
                    break
        
        if len(price_cols) != 4:
            raise ValueError("无法找到完整的价格数据列 (open, high, low, close)")
    
    # 获取价格数据
    open_prices = data[price_cols['open']].values.astype(float)
    high_prices = data[price_cols['high']].values.astype(float)
    low_prices = data[price_cols['low']].values.astype(float)
    close_prices = data[price_cols['close']].values.astype(float)
    volume = data[volume_col].values.astype(float) if volume_col in data.columns else np.ones(len(data)) * 1000
    
    # 默认构建所有特征组
    if feature_groups is None:
        feature_groups = ['overlap', 'momentum', 'volume', 'volatility', 'trend', 'cycle', 'pattern']
    
    print(f"开始构建特征，数据长度: {len(data)}")
    print(f"构建特征组: {feature_groups}")
    
    # 创建一个字典来存储所有特征，最后一次性合并到DataFrame
    features_dict = {}
    
    # 构建重叠指标 (Overlap Studies)
    if 'overlap' in feature_groups:
        print("构建重叠指标...")
        # 移动平均线
        features_dict['MA_5'] = talib.SMA(close_prices, timeperiod=5)
        features_dict['MA_10'] = talib.SMA(close_prices, timeperiod=10)
        features_dict['MA_20'] = talib.SMA(close_prices, timeperiod=20)
        features_dict['MA_30'] = talib.SMA(close_prices, timeperiod=30)
        features_dict['MA_60'] = talib.SMA(close_prices, timeperiod=60)
        
        # 指数移动平均线
        features_dict['EMA_5'] = talib.EMA(close_prices, timeperiod=5)
        features_dict['EMA_10'] = talib.EMA(close_prices, timeperiod=10)
        features_dict['EMA_20'] = talib.EMA(close_prices, timeperiod=20)
        features_dict['EMA_30'] = talib.EMA(close_prices, timeperiod=30)
        
        # 布林带
        upper, middle, lower = talib.BBANDS(close_prices, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
        features_dict['BB_upper'] = upper
        features_dict['BB_middle'] = middle
        features_dict['BB_lower'] = lower
        features_dict['BB_width'] = (upper - lower) / middle  # 布林带宽度
        features_dict['BB_position'] = (close_prices - lower) / (upper - lower)  # 布林带位置
        
        # 其他重叠指标
        features_dict['HT_TRENDLINE'] = talib.HT_TRENDLINE(close_prices)
        features_dict['KAMA'] = talib.KAMA(close_prices, timeperiod=30)
    
    # 构建动量指标 (Momentum Indicators)
    if 'momentum' in feature_groups:
        print("构建动量指标...")
        # RSI
        features_dict['RSI_6'] = talib.RSI(close_prices, timeperiod=6)
        features_dict['RSI_14'] = talib.RSI(close_prices, timeperiod=14)
        features_dict['RSI_24'] = talib.RSI(close_prices, timeperiod=24)
        
        # MACD
        macd, macd_signal, macd_hist = talib.MACD(close_prices, fastperiod=12, slowperiod=26, signalperiod=9)
        features_dict['MACD'] = macd
        features_dict['MACD_signal'] = macd_signal
        features_dict['MACD_hist'] = macd_hist
        features_dict['MACD_diff'] = macd - macd_signal  # MACD差值
        
        # 随机指标
        slowk, slowd = talib.STOCH(high_prices, low_prices, close_prices, 
                                  fastk_period=5, slowk_period=3, slowk_matype=0,
                                  slowd_period=3, slowd_matype=0)
        features_dict['STOCH_K'] = slowk
        features_dict['STOCH_D'] = slowd
        features_dict['STOCH_diff'] = slowk - slowd
        
        # 随机相对强弱指标
        stochrsi_k, stochrsi_d = talib.STOCHRSI(close_prices, timeperiod=14, fastk_period=5, fastd_period=3, fastd_matype=0)
        features_dict['STOCHRSI_fastk'] = stochrsi_k
        features_dict['STOCHRSI_fastd'] = stochrsi_d
        
        # 其他动量指标
        features_dict['MOM'] = talib.MOM(close_prices, timeperiod=10)
        features_dict['ROC'] = talib.ROC(close_prices, timeperiod=10)
        features_dict['ROCR'] = talib.ROCR(close_prices, timeperiod=10)
        features_dict['WILLR'] = talib.WILLR(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['ULTOSC'] = talib.ULTOSC(high_prices, low_prices, close_prices, timeperiod1=7, timeperiod2=14, timeperiod3=28)
        
    # 构建成交量指标 (Volume Indicators)
    if 'volume' in feature_groups:
        print("构建成交量指标...")
        features_dict['AD'] = talib.AD(high_prices, low_prices, close_prices, volume)
        features_dict['ADOSC'] = talib.ADOSC(high_prices, low_prices, close_prices, volume, fastperiod=3, slowperiod=10)
        features_dict['OBV'] = talib.OBV(close_prices, volume)
        
        # 成交量移动平均
        features_dict['VMA_5'] = talib.SMA(volume, timeperiod=5)
        features_dict['VMA_10'] = talib.SMA(volume, timeperiod=10)
        features_dict['VMA_20'] = talib.SMA(volume, timeperiod=20)
        features_dict['VOLUME_RATIO'] = volume / features_dict['VMA_10']  # 成交量比值
    
    # 构建波动率指标 (Volatility Indicators)
    if 'volatility' in feature_groups:
        print("构建波动率指标...")
        features_dict['ATR'] = talib.ATR(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['NATR'] = talib.NATR(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['TRANGE'] = talib.TRANGE(high_prices, low_prices, close_prices)
        
        # 价格波动率
        features_dict['STDDEV'] = talib.STDDEV(close_prices, timeperiod=14, nbdev=1)
        features_dict['VAR'] = talib.VAR(close_prices, timeperiod=14, nbdev=1)
    
    # 构建趋势指标 (Trend Indicators)
    if 'trend' in feature_groups:
        print("构建趋势指标...")
        features_dict['ADX'] = talib.ADX(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['ADXR'] = talib.ADXR(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['APO'] = talib.APO(close_prices, fastperiod=12, slowperiod=26, matype=0)
        features_dict['PPO'] = talib.PPO(close_prices, fastperiod=12, slowperiod=26, matype=0)
        features_dict['PLUS_DI'] = talib.PLUS_DI(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['MINUS_DI'] = talib.MINUS_DI(high_prices, low_prices, close_prices, timeperiod=14)
        features_dict['PLUS_DM'] = talib.PLUS_DM(high_prices, low_prices, timeperiod=14)
        features_dict['MINUS_DM'] = talib.MINUS_DM(high_prices, low_prices, timeperiod=14)
        features_dict['DX'] = talib.DX(high_prices, low_prices, close_prices, timeperiod=14)
    
    # 构建周期指标 (Cycle Indicators)
    if 'cycle' in feature_groups:
        print("构建周期指标...")
        features_dict['HT_DCPERIOD'] = talib.HT_DCPERIOD(close_prices)
        features_dict['HT_DCPHASE'] = talib.HT_DCPHASE(close_prices)
        phasor_inphase, phasor_quadrature = talib.HT_PHASOR(close_prices)
        features_dict['HT_PHASOR_inphase'] = phasor_inphase
        features_dict['HT_PHASOR_quadrature'] = phasor_quadrature
        sine, leadsine = talib.HT_SINE(close_prices)
        features_dict['HT_SINE_sine'] = sine
        features_dict['HT_SINE_leadsine'] = leadsine
        features_dict['HT_TRENDMODE'] = talib.HT_TRENDMODE(close_prices)
    
    # 构建价格模式指标 (Pattern Recognition) - 分批处理以避免性能问题
    if 'pattern' in feature_groups:
        print("构建价格模式指标...")
        # 由于模式识别特征较多，分批添加
        pattern_functions = {
            'CDL2CROWS': lambda: talib.CDL2CROWS(open_prices, high_prices, low_prices, close_prices),
            'CDL3BLACKCROWS': lambda: talib.CDL3BLACKCROWS(open_prices, high_prices, low_prices, close_prices),
            'CDL3INSIDE': lambda: talib.CDL3INSIDE(open_prices, high_prices, low_prices, close_prices),
            'CDL3LINESTRIKE': lambda: talib.CDL3LINESTRIKE(open_prices, high_prices, low_prices, close_prices),
            'CDL3OUTSIDE': lambda: talib.CDL3OUTSIDE(open_prices, high_prices, low_prices, close_prices),
            'CDL3STARSINSOUTH': lambda: talib.CDL3STARSINSOUTH(open_prices, high_prices, low_prices, close_prices),
            'CDL3WHITESOLDIERS': lambda: talib.CDL3WHITESOLDIERS(open_prices, high_prices, low_prices, close_prices),
            'CDLABANDONEDBABY': lambda: talib.CDLABANDONEDBABY(open_prices, high_prices, low_prices, close_prices, penetration=0.3),
            'CDLADVANCEBLOCK': lambda: talib.CDLADVANCEBLOCK(open_prices, high_prices, low_prices, close_prices),
            'CDLBELTHOLD': lambda: talib.CDLBELTHOLD(open_prices, high_prices, low_prices, close_prices),
            'CDLBREAKAWAY': lambda: talib.CDLBREAKAWAY(open_prices, high_prices, low_prices, close_prices),
            'CDLCLOSINGMARUBOZU': lambda: talib.CDLCLOSINGMARUBOZU(open_prices, high_prices, low_prices, close_prices),
            'CDLCONCEALBABYSWALL': lambda: talib.CDLCONCEALBABYSWALL(open_prices, high_prices, low_prices, close_prices),
            'CDLCOUNTERATTACK': lambda: talib.CDLCOUNTERATTACK(open_prices, high_prices, low_prices, close_prices),
            'CDLDARKCLOUDCOVER': lambda: talib.CDLDARKCLOUDCOVER(open_prices, high_prices, low_prices, close_prices, penetration=0.5),
            'CDLDOJI': lambda: talib.CDLDOJI(open_prices, high_prices, low_prices, close_prices),
            'CDLDOJISTAR': lambda: talib.CDLDOJISTAR(open_prices, high_prices, low_prices, close_prices),
            'CDLDRAGONFLYDOJI': lambda: talib.CDLDRAGONFLYDOJI(open_prices, high_prices, low_prices, close_prices),
            'CDLENGULFING': lambda: talib.CDLENGULFING(open_prices, high_prices, low_prices, close_prices),
            'CDLEVENINGDOJISTAR': lambda: talib.CDLEVENINGDOJISTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3),
            'CDLEVENINGSTAR': lambda: talib.CDLEVENINGSTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3),
            'CDLGAPSIDESIDEWHITE': lambda: talib.CDLGAPSIDESIDEWHITE(open_prices, high_prices, low_prices, close_prices),
            'CDLGRAVESTONEDOJI': lambda: talib.CDLGRAVESTONEDOJI(open_prices, high_prices, low_prices, close_prices),
            'CDLHAMMER': lambda: talib.CDLHAMMER(open_prices, high_prices, low_prices, close_prices),
            'CDLHANGINGMAN': lambda: talib.CDLHANGINGMAN(open_prices, high_prices, low_prices, close_prices),
            'CDLHARAMI': lambda: talib.CDLHARAMI(open_prices, high_prices, low_prices, close_prices),
            'CDLHARAMICROSS': lambda: talib.CDLHARAMICROSS(open_prices, high_prices, low_prices, close_prices),
            'CDLHIGHWAVE': lambda: talib.CDLHIGHWAVE(open_prices, high_prices, low_prices, close_prices),
            'CDLHIKKAKE': lambda: talib.CDLHIKKAKE(open_prices, high_prices, low_prices, close_prices),
            'CDLHIKKAKEMOD': lambda: talib.CDLHIKKAKEMOD(open_prices, high_prices, low_prices, close_prices),
            'CDLHOMINGPIGEON': lambda: talib.CDLHOMINGPIGEON(open_prices, high_prices, low_prices, close_prices),
            'CDLIDENTICAL3CROWS': lambda: talib.CDLIDENTICAL3CROWS(open_prices, high_prices, low_prices, close_prices),
            'CDLINNECK': lambda: talib.CDLINNECK(open_prices, high_prices, low_prices, close_prices),
            'CDLINVERTEDHAMMER': lambda: talib.CDLINVERTEDHAMMER(open_prices, high_prices, low_prices, close_prices),
            'CDLKICKING': lambda: talib.CDLKICKING(open_prices, high_prices, low_prices, close_prices),
            'CDLKICKINGBYLENGTH': lambda: talib.CDLKICKINGBYLENGTH(open_prices, high_prices, low_prices, close_prices),
            'CDLLADDERBOTTOM': lambda: talib.CDLLADDERBOTTOM(open_prices, high_prices, low_prices, close_prices),
            'CDLLONGLEGGEDDOJI': lambda: talib.CDLLONGLEGGEDDOJI(open_prices, high_prices, low_prices, close_prices),
            'CDLLONGLINE': lambda: talib.CDLLONGLINE(open_prices, high_prices, low_prices, close_prices),
            'CDLMARUBOZU': lambda: talib.CDLMARUBOZU(open_prices, high_prices, low_prices, close_prices),
            'CDLMATCHINGLOW': lambda: talib.CDLMATCHINGLOW(open_prices, high_prices, low_prices, close_prices),
            'CDLMATHOLD': lambda: talib.CDLMATHOLD(open_prices, high_prices, low_prices, close_prices, penetration=0.5),
            'CDLMORNINGDOJISTAR': lambda: talib.CDLMORNINGDOJISTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3),
            'CDLMORNINGSTAR': lambda: talib.CDLMORNINGSTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3),
            'CDLONNECK': lambda: talib.CDLONNECK(open_prices, high_prices, low_prices, close_prices),
            'CDLPIERCING': lambda: talib.CDLPIERCING(open_prices, high_prices, low_prices, close_prices),
            'CDLRICKSHAWMAN': lambda: talib.CDLRICKSHAWMAN(open_prices, high_prices, low_prices, close_prices),
            'CDLRISEFALL3METHODS': lambda: talib.CDLRISEFALL3METHODS(open_prices, high_prices, low_prices, close_prices),
            'CDLSEPARATINGLINES': lambda: talib.CDLSEPARATINGLINES(open_prices, high_prices, low_prices, close_prices),
            'CDLSHOOTINGSTAR': lambda: talib.CDLSHOOTINGSTAR(open_prices, high_prices, low_prices, close_prices),
            'CDLSHORTLINE': lambda: talib.CDLSHORTLINE(open_prices, high_prices, low_prices, close_prices),
            'CDLSPINNINGTOP': lambda: talib.CDLSPINNINGTOP(open_prices, high_prices, low_prices, close_prices),
            'CDLSTALLEDPATTERN': lambda: talib.CDLSTALLEDPATTERN(open_prices, high_prices, low_prices, close_prices),
            'CDLSTICKSANDWICH': lambda: talib.CDLSTICKSANDWICH(open_prices, high_prices, low_prices, close_prices),
            'CDLTAKURI': lambda: talib.CDLTAKURI(open_prices, high_prices, low_prices, close_prices),
            'CDLTASUKIGAP': lambda: talib.CDLTASUKIGAP(open_prices, high_prices, low_prices, close_prices),
            'CDLTHRUSTING': lambda: talib.CDLTHRUSTING(open_prices, high_prices, low_prices, close_prices),
            'CDLTRISTAR': lambda: talib.CDLTRISTAR(open_prices, high_prices, low_prices, close_prices),
            'CDLUNIQUE3RIVER': lambda: talib.CDLUNIQUE3RIVER(open_prices, high_prices, low_prices, close_prices),
            'CDLUPSIDEGAP2CROWS': lambda: talib.CDLUPSIDEGAP2CROWS(open_prices, high_prices, low_prices, close_prices),
            'CDLXSIDEGAP3METHODS': lambda: talib.CDLXSIDEGAP3METHODS(open_prices, high_prices, low_prices, close_prices),
        }
        
        # 批量添加模式识别特征
        for pattern_name, pattern_func in pattern_functions.items():
            try:
                features_dict[pattern_name] = pattern_func()
            except Exception as e:
                print(f"警告: 计算 {pattern_name} 时出错: {e}")
                features_dict[pattern_name] = np.full(len(data), 0)  # 用0填充
    
    # 自定义特征 ====================
    print("计算自定义特征...")
    df['return'] = np.log(df['close'] / df['close'].shift(1))
    df['macd'] = macd
    df['rsi'] = talib.RSI(close_prices, timeperiod=14) 
    # 基础收益率
    data['return'] = np.log(df['close'] / df['close'].shift(1))
    data['return'] = np.log(df['close'] / df['close'].shift(1))
    data['gap'] = np.log(df['open'] / df['close'].shift(1))
    
    #市场情绪
    # data['hot'] = df['up_count'] / df['down_count']
    # K线形态特征
    data['body'] = df['close'] - df['open']
    data['body_abs'] = np.abs(data['body'])
    data['body_ratio'] = data['body_abs'] / (df['high'] - df['low'] + 1e-8)
    
    data['upper_shadow'] = df['high'] - np.maximum(df['open'], df['close'])
    data['lower_shadow'] = np.minimum(df['open'], df['close']) - df['low']
    data['upper_ratio'] = data['upper_shadow'] / (df['high'] - df['low'] + 1e-8)
    data['lower_ratio'] = data['lower_shadow'] / (df['high'] - df['low'] + 1e-8)
    
    # 真实波动率
    true_range = np.maximum(
        df['high'] - df['low'],
        np.abs(df['high'] - df['close'].shift(1)),
        np.abs(df['low'] - df['close'].shift(1))
    )
    data['true_range'] = true_range
    data['true_range_ratio'] = true_range / df['close']
    
    # 量价关系
    data['volume_change'] = np.log(df['amount'] / df['amount'].shift(1))
    data['price_volume_corr'] = data['return'].rolling(10).corr(data['volume_change'])
    
    # 滚动统计
    for window in [5, 10, 20]:
        data[f'return_mean_{window}'] = df['return'].rolling(window).mean()
        data[f'return_std_{window}'] = df['return'].rolling(window).std()
        data[f'return_skew_{window}'] = df['return'].rolling(window).skew()
        data[f'return_kurt_{window}'] = df['return'].rolling(window).kurt()
        data[f'volume_mean_{window}'] = df['amount'].rolling(window).mean()
    
    # 滞后特征
    for lag in [1, 2, 3, 5]:
        data[f'return_lag{lag}'] = df['return'].shift(lag)
        data[f'rsi_lag{lag}'] = df['rsi'].shift(lag)
        data[f'macd_lag{lag}'] = df['macd'].shift(lag)
        data[f'volume_lag{lag}'] = df['amount'].shift(lag)
    
    # # 位置关系特征
    # df['close_sma20_ratio'] = df['close'] / df['sma_20']
    # df['close_bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-8)
    
    # # 方向特征
    # df['direction'] = (df['close'] > df['open']).astype(int)
    # df['trend_up'] = (df['close'] > df['sma_20']).astype(int)
    # df['ma5_above_ma20'] = (df['sma_5'] > df['sma_20']).astype(int)

    # 添加自定义特征-------
    # 价格相关特征
    features_dict['HIGH_LOW_RATIO'] = high_prices / low_prices
    features_dict['CLOSE_OPEN_RATIO'] = close_prices / open_prices
    features_dict['HIGH_CLOSE_RATIO'] = high_prices / close_prices
    features_dict['LOW_CLOSE_RATIO'] = low_prices / close_prices
    
    # 价格变化率
    features_dict['PRICE_CHANGE'] = np.concatenate([[np.nan], (close_prices[1:] - close_prices[:-1]) / close_prices[:-1]])
    features_dict['HIGH_LOW_PCT'] = (high_prices - low_prices) / close_prices
    features_dict['HIGH_CLOSE_PCT'] = (high_prices - close_prices) / close_prices
    features_dict['CLOSE_LOW_PCT'] = (close_prices - low_prices) / close_prices
    
    # 移动平均线之间的关系
    if 'MA_5' in features_dict and 'MA_10' in features_dict:
        features_dict['MA_5_10_RATIO'] = features_dict['MA_5'] / features_dict['MA_10']
        features_dict['MA_5_10_DIFF'] = features_dict['MA_5'] - features_dict['MA_10']
    
    if 'MA_20' in features_dict and 'MA_60' in features_dict:
        features_dict['MA_20_60_RATIO'] = features_dict['MA_20'] / features_dict['MA_60']
        features_dict['MA_20_60_DIFF'] = features_dict['MA_20'] - features_dict['MA_60']
    
    # 波动率相关特征
    if 'STDDEV' in features_dict:
        features_dict['VOLATILITY_RATIO'] = features_dict['STDDEV'] / close_prices
    
    # 将所有特征一次性添加到DataFrame
    features_df = pd.DataFrame(features_dict, index=data.index)
    
    # 合并特征到原始数据
    result_data = pd.concat([data, features_df], axis=1)
    
    # 处理缺失值
    print("处理缺失值...")
    result_data = result_data.replace([np.inf, -np.inf], np.nan)
    result_data = result_data.bfill().ffill()
    
    print(f"特征构建完成，共生成 {len(result_data.columns)} 个特征")
    return result_data

In [ ]:
ddf  = create_features(df,
                     price_cols={'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close'},
                     volume_col='volume',
                    #  volume_col='amount',
                     date_col='date',
                    #  feature_groups=['overlap', 'momentum', 'volume'])
                     feature_groups=['overlap', 'momentum', 'volume', 'volatility', 'trend', 'cycle', 'pattern'])

In [ ]:
import optuna
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def objective(trial, X, y):
    # 定义搜索空间
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": "gbtree",
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "n_estimators": 1000,
        "random_state": 42,
        "verbosity": 0,  # <-- 关键：控制日志输出
        "device": "gpu",  # 如果有GPU可用
        "tree_method": "hist"  # 加速
    }

    # 时间序列交叉验证（避免未来信息泄露）
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBClassifier(**params, early_stopping_rounds=50)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        y_pred = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred)
        scores.append(score)

    return np.mean(scores)


In [ ]:
# 构建二分类标签
feature_columns = ddf.drop(columns='date').columns
X = ddf[feature_columns]
y = (ddf['return'].rolling(13).sum().shift(-13).ffill() > 0.10).astype(int)

In [ ]:
total_size = len(ddf)
train_end_idx = int(0.7 * total_size)
val_end_idx = int(0.85 * total_size)


X_train = X.iloc[:train_end_idx]
X_val = X.iloc[train_end_idx:val_end_idx]
X_test = X.iloc[val_end_idx:]
y_train = y.iloc[:train_end_idx]
y_val = y.iloc[train_end_idx:val_end_idx]
y_test = y.iloc[val_end_idx:]

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
# 启动优化
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=100, show_progress_bar=True)

best_params = study.best_params
print("Best params:", best_params)

#### 基于 XGBoost 内置重要性（初步筛选）

In [ ]:
model = XGBClassifier(**best_params)
model.fit(X_train, y_train)
importances = model.feature_importances_

In [ ]:
import shap
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# 计算绝对SHAP均值作为重要性
shap_importance = np.mean(np.abs(shap_values), axis=0)
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'shap_importance': shap_importance
}).sort_values('shap_importance', ascending=False)

# 保留 top-k 特征（如 top 30）
top_features = feature_importance_df.head(30)['feature'].tolist()
X_selected = X[top_features]

In [ ]:
import plotly.express as px

fig = px.bar(feature_importance_df.head(20), 
             x='shap_importance', y='feature', 
             orientation='h', title='Top 20 Features by SHAP Importance')
fig.show()

In [ ]:
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0,:], X.iloc[0,:], matplotlib=False)